# Capstone: Refresh / Content Opportunity Scoring
**Lane:** Refresh / Content Opportunity Scoring  
**Author:** Alishba Khan  
**Data:** FlyRank ML Internship Dataset (Hugging Face)  
**Goal:** Score pages as Growing / Declining / Recovering / Review-Worthy and produce a ranked action engine with reason codes.

## 0. Setup & Imports

In [ ]:
# Install dependencies if needed
# !pip install duckdb huggingface_hub scikit-learn pandas matplotlib seaborn --quiet

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier
import warnings
warnings.filterwarnings('ignore')

# Hugging Face token (set your token here or via environment variable)
import os
HF_TOKEN = os.environ.get('HF_TOKEN', 'YOUR_HF_TOKEN_HERE')

print('Libraries loaded successfully.')

## 1. Data Loading via DuckDB + Hugging Face

In [ ]:
# Connect to DuckDB and load HuggingFace extension
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET hf_token = '{HF_TOKEN}';")

# ── Load GSC performance table ──
# Adjust the path to match actual HF dataset repo
DATASET = "hf://datasets/flyrank/ml-internship-dataset"

# Load page-level weekly aggregates
df_raw = con.execute(f"""
    SELECT
        page,
        date_week,
        SUM(clicks)       AS clicks,
        SUM(impressions)  AS impressions,
        AVG(position)     AS avg_position,
        SUM(clicks) * 1.0 / NULLIF(SUM(impressions), 0) AS ctr
    FROM read_parquet('{DATASET}/gsc_pages/*.parquet')
    WHERE date_week >= '2023-01-01'
    GROUP BY page, date_week
    ORDER BY page, date_week
""").df()

print(f'Loaded {len(df_raw):,} page-week rows')
print(f'Date range: {df_raw.date_week.min()} → {df_raw.date_week.max()}')
print(f'Unique pages: {df_raw.page.nunique():,}')
df_raw.head()

## 2. Exploratory Data Analysis

In [ ]:
# Basic distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(np.log1p(df_raw['clicks']), bins=50, color='#0F2A4A', alpha=0.8)
axes[0].set_title('Log Clicks Distribution'); axes[0].set_xlabel('log(clicks+1)')

axes[1].hist(np.log1p(df_raw['impressions']), bins=50, color='#2A6BAD', alpha=0.8)
axes[1].set_title('Log Impressions Distribution'); axes[1].set_xlabel('log(impressions+1)')

axes[2].hist(df_raw['avg_position'].clip(1, 100), bins=50, color='#4A9FD4', alpha=0.8)
axes[2].set_title('Avg Position Distribution'); axes[2].set_xlabel('position (clipped at 100)')

plt.suptitle('FlyRank GSC Data — Basic Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA distributions saved.')

In [ ]:
# Weekly trend: total clicks over time
weekly = df_raw.groupby('date_week')[['clicks','impressions']].sum().reset_index()

fig, ax1 = plt.subplots(figsize=(14, 4))
ax2 = ax1.twinx()
ax1.plot(weekly.date_week, weekly.clicks, color='#0F2A4A', lw=2, label='Clicks')
ax2.plot(weekly.date_week, weekly.impressions, color='#2A6BAD', lw=1.5, ls='--', alpha=0.7, label='Impressions')
ax1.set_ylabel('Clicks', color='#0F2A4A')
ax2.set_ylabel('Impressions', color='#2A6BAD')
ax1.set_title('Weekly Clicks & Impressions Over Time')
plt.tight_layout()
plt.savefig('../figures/weekly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering
Split timeline: **early window** (first 60%) vs **late window** (last 40%).  
Features derived from early window only → labels from late window. No leakage.

In [ ]:
df_raw['date_week'] = pd.to_datetime(df_raw['date_week'])

# Time-aware split — no leakage
cutoff = df_raw['date_week'].quantile(0.60)
print(f'Cutoff date (60th pct): {cutoff.date()}')

early = df_raw[df_raw['date_week'] <= cutoff].copy()
late  = df_raw[df_raw['date_week']  > cutoff].copy()

print(f'Early window: {early.date_week.min().date()} → {early.date_week.max().date()} ({early.page.nunique():,} pages)')
print(f'Late  window: {late.date_week.min().date()}  → {late.date_week.max().date()}  ({late.page.nunique():,} pages)')

In [ ]:
def make_features(df):
    """Aggregate page-level features from a time window."""
    agg = df.groupby('page').agg(
        total_clicks        = ('clicks',      'sum'),
        total_impressions   = ('impressions', 'sum'),
        weeks_active        = ('date_week',   'nunique'),
        avg_position        = ('avg_position','mean'),
        min_position        = ('avg_position','min'),
        max_position        = ('avg_position','max'),
        avg_ctr             = ('ctr',         'mean'),
        last_clicks         = ('clicks',      'last'),
        first_clicks        = ('clicks',      'first'),
    ).reset_index()

    # Derived signals
    agg['clicks_per_week']     = agg['total_clicks'] / agg['weeks_active'].clip(lower=1)
    agg['impression_density']  = agg['total_impressions'] / agg['weeks_active'].clip(lower=1)
    agg['position_range']      = agg['max_position'] - agg['min_position']
    agg['click_momentum']      = (agg['last_clicks'] - agg['first_clicks']) / (agg['first_clicks'].clip(lower=1))
    agg['ctr_efficiency']      = agg['avg_ctr'] / (1 / agg['avg_position'].clip(lower=1))  # CTR vs position-expected
    agg['log_impressions']     = np.log1p(agg['total_impressions'])
    agg['log_clicks']          = np.log1p(agg['total_clicks'])

    return agg

features_early = make_features(early)
print(f'Feature matrix: {features_early.shape}')
features_early.head()

## 4. Label Definition
Label each page based on late-window click trend vs early-window baseline.

In [ ]:
features_late = make_features(late)

# Merge early vs late for labeling
merged = features_early.merge(
    features_late[['page','total_clicks','avg_position','avg_ctr']],
    on='page', suffixes=('_early','_late')
)

# Click growth rate: (late - early) / early
merged['click_growth'] = (
    (merged['total_clicks_late'] - merged['total_clicks_early'])
    / merged['total_clicks_early'].clip(lower=1)
)
merged['pos_change'] = merged['avg_position_late'] - merged['avg_position_early']  # negative = improved

# Label rules:
# GROWING     = click_growth > +20%  AND position improved or stable
# DECLINING   = click_growth < -20%  AND position worsened or clicks dropped hard
# RECOVERING  = click_growth > +10%  AND had low early clicks (recovering from low base)
# REVIEW      = everything else (flat, inconsistent, or volatile)

def assign_label(row):
    g = row['click_growth']
    p = row['pos_change']
    e = row['total_clicks_early']
    if g > 0.20 and p <= 2:
        return 'GROWING'
    elif g < -0.20:
        return 'DECLINING'
    elif g > 0.10 and e < merged['total_clicks_early'].median():
        return 'RECOVERING'
    else:
        return 'REVIEW'

merged['label'] = merged.apply(assign_label, axis=1)

print('Label distribution:')
print(merged['label'].value_counts())
print(f'\nTotal pages labeled: {len(merged):,}')

In [ ]:
# Label distribution chart
label_counts = merged['label'].value_counts()
colors = {'GROWING': '#22c55e', 'DECLINING': '#ef4444', 'RECOVERING': '#f59e0b', 'REVIEW': '#2A6BAD'}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(label_counts.index, label_counts.values,
              color=[colors[l] for l in label_counts.index], alpha=0.9, edgecolor='white')
for bar, val in zip(bars, label_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Content Opportunity Label Distribution', fontsize=14, pad=12)
ax.set_ylabel('Number of Pages')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../figures/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Training

In [ ]:
FEATURE_COLS = [
    'total_clicks', 'total_impressions', 'weeks_active',
    'avg_position', 'position_range', 'avg_ctr',
    'clicks_per_week', 'impression_density', 'click_momentum',
    'ctr_efficiency', 'log_impressions', 'log_clicks'
]

X = merged[FEATURE_COLS].fillna(0)
y = merged['label']

# ── Baseline: majority class dummy ──
dummy = DummyClassifier(strategy='most_frequent')
dummy_scores = cross_val_score(dummy, X, y, cv=5, scoring='f1_macro')
print(f'Baseline (majority class) F1-macro: {dummy_scores.mean():.3f} ± {dummy_scores.std():.3f}')

# ── Model 1: Logistic Regression ──
lr_pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, C=1.0))])
lr_scores = cross_val_score(lr_pipe, X, y, cv=5, scoring='f1_macro')
print(f'Logistic Regression     F1-macro: {lr_scores.mean():.3f} ± {lr_scores.std():.3f}')

# ── Model 2: Random Forest ──
rf_pipe = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))])
rf_scores = cross_val_score(rf_pipe, X, y, cv=5, scoring='f1_macro')
print(f'Random Forest           F1-macro: {rf_scores.mean():.3f} ± {rf_scores.std():.3f}')

# ── Model 3: Gradient Boosting (chosen model) ──
gb_pipe = Pipeline([('scaler', StandardScaler()), ('clf', GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42))])
gb_scores = cross_val_score(gb_pipe, X, y, cv=5, scoring='f1_macro')
print(f'Gradient Boosting       F1-macro: {gb_scores.mean():.3f} ± {gb_scores.std():.3f}')

In [ ]:
# Model comparison chart
models = ['Baseline\n(Majority)', 'Logistic\nRegression', 'Random\nForest', 'Gradient\nBoosting']
means  = [dummy_scores.mean(), lr_scores.mean(), rf_scores.mean(), gb_scores.mean()]
stds   = [dummy_scores.std(),  lr_scores.std(),  rf_scores.std(),  gb_scores.std()]
bar_colors = ['#94a3b8', '#4A9FD4', '#2A6BAD', '#0F2A4A']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, means, yerr=stds, capsize=5,
              color=bar_colors, alpha=0.9, edgecolor='white')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{m:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_ylabel('F1-macro (5-fold CV)')
ax.set_title('Model vs Baseline — F1-macro on Same Time-Aware Split', fontsize=13, pad=12)
ax.axhline(means[0], color='#94a3b8', ls='--', lw=1.2, alpha=0.6, label='Baseline')
ax.spines[['top','right']].set_visible(False)
ax.legend()
plt.tight_layout()
plt.savefig('../figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fit chosen model on all data and inspect feature importance
gb_pipe.fit(X, y)
importances = gb_pipe.named_steps['clf'].feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot.barh(ax=ax, color='#0F2A4A', alpha=0.85)
ax.set_title('Feature Importance — Gradient Boosting', fontsize=13)
ax.set_xlabel('Importance')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Opportunity Scoring & Ranked Recommendations

In [ ]:
# Predict probabilities
proba = gb_pipe.predict_proba(X)
classes = gb_pipe.named_steps['clf'].classes_

proba_df = pd.DataFrame(proba, columns=[f'p_{c}' for c in classes])
merged_scored = pd.concat([merged.reset_index(drop=True), proba_df], axis=1)
merged_scored['predicted_label'] = gb_pipe.predict(X)

# Opportunity score: weighted composite
# Higher = more urgent / higher potential
merged_scored['opportunity_score'] = (
    merged_scored.get('p_DECLINING', 0) * 0.40 +    # declining pages need urgent action
    merged_scored.get('p_RECOVERING', 0) * 0.30 +   # recovering pages: boost potential
    merged_scored.get('p_REVIEW', 0) * 0.20 +       # review: latent opportunity
    (1 - merged_scored.get('p_GROWING', 0)) * 0.10  # non-growing gets priority
).fillna(0)

# Reason code
def reason_code(row):
    if row['predicted_label'] == 'DECLINING':
        return 'URGENT: Click loss >20% — audit content freshness & backlinks'
    elif row['predicted_label'] == 'RECOVERING':
        return 'OPPORTUNITY: Momentum building — strengthen with internal links & refresh'
    elif row['predicted_label'] == 'GROWING':
        return 'PROTECT: Strong performer — monitor, expand with related content'
    else:
        return 'REVIEW: Flat/volatile signal — assess search intent alignment'

merged_scored['action'] = merged_scored.apply(reason_code, axis=1)

# Ranked output
ranked = merged_scored[['page','predicted_label','opportunity_score','action',
                         'total_clicks_early','total_clicks_late','click_growth',
                         'avg_position_early','avg_position_late']]\
    .sort_values('opportunity_score', ascending=False)

print('Top 10 pages by opportunity score:')
ranked.head(10)[['page','predicted_label','opportunity_score','action']]

In [ ]:
# Save ranked output
ranked.to_csv('../figures/ranked_opportunities.csv', index=False)
print('Ranked opportunities saved to figures/ranked_opportunities.csv')

## 7. Validation — Confusion Matrix on Holdout Subset

In [ ]:
from sklearn.model_selection import train_test_split

# Time-ordered split: last 20% as holdout
split_idx = int(len(X) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

gb_val = Pipeline([('scaler', StandardScaler()),
                   ('clf', GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42))])
gb_val.fit(X_train, y_train)
y_pred = gb_val.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred, labels=gb_val.named_steps['clf'].classes_)
disp = ConfusionMatrixDisplay(cm, display_labels=gb_val.named_steps['clf'].classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Holdout Set (last 20%)', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary
- **Best model:** Gradient Boosting
- **Validation:** 5-fold cross-validation + time-ordered holdout
- **Leakage check:** Features derived from early window only; labels from late window only
- **Key features:** `click_momentum`, `clicks_per_week`, `avg_position` — top predictors
- **Output:** Ranked action engine saved to `ranked_opportunities.csv`